In [1]:
import os

python_path = r"C:\Users\jatin\anaconda3\envs\spark311\python.exe"

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

spark session building

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Cross_System_Monitoring")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

read data 

In [3]:
crm = spark.read.format("delta").load("../silver/crm")

billing = spark.read.format("delta").load("../silver/billing")

analytics = spark.read.format("delta").load("../silver/analytics")

trust = spark.read.format("delta").load("../gold/trust_score")

missing = spark.read.format("delta").load("../gold/missing_records")

duplicates = spark.read.format("delta").load("../gold/duplicates")

drift = spark.read.format("delta").load("../gold/drift_report")

printing data from delta table

In [4]:
crm.show(5)

billing.show(5)

analytics.show(5)

trust.show()

missing.show()

duplicates.show()

drift.show()

+-----------+-----------+--------------------+-----------+-------------+
|customer_id|       name|               email|signup_date|         city|
+-----------+-----------+--------------------+-----------+-------------+
|  CRM000001|Rahul Singh|rahul.singh56@out...| 2024-06-20|        Delhi|
|  CRM000002|Aarav Joshi|aarav.joshi375@ho...| 2022-06-28|        Surat|
|  CRM000003| Rohan Shah|rohan.shah787@yah...| 2024-02-19|Visakhapatnam|
|  CRM000004|Arjun Singh|arjun.singh555@ou...| 2024-02-20|       Mumbai|
|  CRM000005| Amit Sinha|amit.sinha514@yah...| 2022-09-26|    Hyderabad|
+-----------+-----------+--------------------+-----------+-------------+
only showing top 5 rows

+--------------+-----------+-------+----------------+---------+
|transaction_id|customer_id| amount|transaction_date|   status|
+--------------+-----------+-------+----------------+---------+
|    TXN0000001|  CRM001845| 138.25|      2023-07-27|completed|
|    TXN0000002|  CRM008227|1121.36|      2023-02-09|completed

summary of delta table

In [5]:
crm_count = crm.count()

billing_count = billing.count()

analytics_count = analytics.count()

missing_count = missing.count()

duplicate_count = duplicates.count()

drift_count = drift.count()

trust table

In [6]:
trust.show()

trust_score = trust.collect()[0]["trust_score"]

+-----------+
|trust_score|
+-----------+
|       36.6|
+-----------+



creating dashboard

In [7]:
dashboard = [
    ("CRM Records", float(crm_count)),
    ("Billing Records", float(billing_count)),
    ("Analytics Records", float(analytics_count)),
    ("Missing Records", float(missing_count)),
    ("Duplicate Records", float(duplicate_count)),
    ("Drift Records", float(drift_count)),
    ("Trust Score", float(trust_score))   ]

dashboard_df = spark.createDataFrame( dashboard, ["Metric", "Value"])

dashboard_df.show(truncate=False)


+-----------------+-------+
|Metric           |Value  |
+-----------------+-------+
|CRM Records      |10500.0|
|Billing Records  |11680.0|
|Analytics Records|912.0  |
|Missing Records  |3959.0 |
|Duplicate Records|0.0    |
|Drift Records    |912.0  |
|Trust Score      |36.6   |
+-----------------+-------+



change dashboard type int to float

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("Metric", StringType(), True),
    StructField("Value", DoubleType(), True)
])

dashboard_df = spark.createDataFrame(dashboard, schema)

dashboard_df.show(truncate=False)

+-----------------+-------+
|Metric           |Value  |
+-----------------+-------+
|CRM Records      |10500.0|
|Billing Records  |11680.0|
|Analytics Records|912.0  |
|Missing Records  |3959.0 |
|Duplicate Records|0.0    |
|Drift Records    |912.0  |
|Trust Score      |36.6   |
+-----------------+-------+



saving dashboard data into delta table

In [59]:
dashboard_df.write \
.format("delta") \
.mode("overwrite") \
.save("../gold/dashboard_data")

In [9]:
dashboard = spark.read.format("delta").load("../gold/dashboard_data")

dashboard.show(truncate=False)

+-----------------+-------+
|Metric           |Value  |
+-----------------+-------+
|Analytics Records|912.0  |
|Duplicate Records|0.0    |
|Missing Records  |3959.0 |
|Billing Records  |11680.0|
|Drift Records    |912.0  |
|CRM Records      |10500.0|
|Trust Score      |36.6   |
+-----------------+-------+



creating dashboard into csv file

In [11]:
dashboard_df.toPandas().to_csv( "../dashboard/dashboard.csv", index=False)

In [12]:
print("Dashboard Created Successfully")

print("Total Metrics :", dashboard.count())

Dashboard Created Successfully
Total Metrics : 7
